In [1]:
from datetime import datetime, timedelta
from utils.spark_session import get_sparkSession 
from utils.utilities import generate_date

In [2]:
from pyspark.sql.functions import current_timestamp, lit, expr, to_date, when, lower, upper, trim

In [3]:
spark = get_sparkSession("Load silver.customer")

In [4]:
print("SPARK-APP : spark-UI - " + spark.sparkContext.uiWebUrl)

SPARK-APP : spark-UI - http://ba7b2bbb9aec:4040


In [5]:
from utils.load_controller import insert_control_logs, get_max_loadTimestamp

In [6]:
_schema_name = "silver"
_table_name = "customer"
_fullname = f"{_schema_name}.{_table_name}"
_script_name = "silver_customer.ipynb"
_bronze_table = "bronze.customer"

print(f"SPARK-APP : Loading started for {_fullname}")

SPARK-APP : Loading started for silver.customer


In [7]:
#spark config

spark.conf.set("spark.sql.shuffle.partitions",16)


In [8]:
#Read get_max_loadTimestamp

max_timestamp = get_max_loadTimestamp(spark, _schema_name, _table_name)


In [9]:
#Reading from bronze.raw_customer and de-duplicating the data

df = spark.read.table(_bronze_table) \
          .where(f"created_on > to_timestamp('{max_timestamp}')")

print(f"SPARK-APP: Bronze Table Data count : {df.count()}")

df_dedup = df.withColumn("_rn", expr("row_number() over (partition by customer_id, store, channel, effective_start_date order by created_on desc)")) \
             .where("_rn = 1") \
             .drop("_rn")

loaded_count = df_dedup.count()

print(f"SPARK-APP: Table count after de-dupe : {loaded_count}")

SPARK-APP: Bronze Table Data count : 5
SPARK-APP: Table count after de-dupe : 5


In [10]:
df.show(10)
df.printSchema()


+-----------+------+-------+----------+-----------+---------+------------+------------------+------+-------------+--------------------+----------+----------+----------------+--------------------+------------------+--------------------+------------------+
|customer_id| store|channel|first_name|middle_name|last_name|phone_number|             email|gender|date_of_birth|effective_start_date|is_current|is_deleted|     source_file|          created_on|        created_by|         modified_on|       modified_by|
+-----------+------+-------+----------+-----------+---------+------------+------------------+------+-------------+--------------------+----------+----------+----------------+--------------------+------------------+--------------------+------------------+
|    CUST001|AMZ_US|    MKT|      John|       null|      Doe|  9991112222|    john@gmail.com|  Male|   1988-05-10|          2024-01-01|     false|     false|dim_customer.csv|2026-01-29 00:02:...|raw_customer.ipynb|2026-01-29 00:02:...|

In [11]:
#Formating the data

df_ld = df_dedup.withColumn("date_of_birth", to_date("date_of_birth", 'yyyy-MM-dd')) \
                .withColumn("effective_start_date", to_date("effective_start_date", 'yyyy-MM-dd')) \
                .withColumn("is_current", when(lower("is_current") == "true", True).otherwise(False)) \
                .withColumn("is_deleted", when(lower("is_deleted") == "true", True).otherwise(False))
                

In [12]:
#Standardizing the codes and types

df_ld = df_ld.withColumn("customer_id", upper(trim("customer_id"))) \
             .withColumn("store", upper(trim("store"))) \
             .withColumn("channel", upper(trim("channel")))

In [13]:
#Adding audit columns

df_ld = df_ld.withColumn("created_on", current_timestamp()) \
             .withColumn("created_by", lit(_script_name)) \
             .withColumn("modified_on", current_timestamp()) \
             .withColumn("modified_by", lit(_script_name))

In [14]:
df_ld.show(10)
df_ld.printSchema()

+-----------+------+-------+----------+-----------+---------+------------+------------------+------+-------------+--------------------+----------+----------+----------------+--------------------+--------------------+--------------------+--------------------+
|customer_id| store|channel|first_name|middle_name|last_name|phone_number|             email|gender|date_of_birth|effective_start_date|is_current|is_deleted|     source_file|          created_on|          created_by|         modified_on|         modified_by|
+-----------+------+-------+----------+-----------+---------+------------+------------------+------+-------------+--------------------+----------+----------+----------------+--------------------+--------------------+--------------------+--------------------+
|    CUST001|AMZ_US|    MKT|      John|       null|      Doe|  9991112222|    john@gmail.com|  Male|   1988-05-10|          2024-01-01|     false|     false|dim_customer.csv|2026-01-29 00:51:...|silver_customer.i...|2026-01

In [15]:
#Writing to Table and log data to load_controller

_data = []

df_ld.write \
     .format("delta") \
     .mode("overwrite") \
     .saveAsTable(_fullname)
    
_data.append([_schema_name, _schema_name, _table_name, "full-load", "overwrite", str(datetime.now()), "success", loaded_count, _script_name, _script_name])

insert_control_logs(spark, _data)
    
print(f"SPARK-APP: Data written to {_schema_name}.{_table_name} and load_controller")

SPARK-APP: Data written to silver.customer and load_controller


In [16]:
spark.read.table("silver.customer").show()

+-----------+------+-------+----------+-----------+---------+------------+------------------+------+-------------+--------------------+----------+----------+----------------+--------------------+--------------------+--------------------+--------------------+
|customer_id| store|channel|first_name|middle_name|last_name|phone_number|             email|gender|date_of_birth|effective_start_date|is_current|is_deleted|     source_file|          created_on|          created_by|         modified_on|         modified_by|
+-----------+------+-------+----------+-----------+---------+------------+------------------+------+-------------+--------------------+----------+----------+----------------+--------------------+--------------------+--------------------+--------------------+
|    CUST001|AMZ_US|    MKT|      John|       null|      Doe|  9991112222|    john@gmail.com|  Male|   1988-05-10|          2024-01-01|     false|     false|dim_customer.csv|2026-01-29 00:52:...|silver_customer.i...|2026-01

In [17]:
spark.sql(f"select * from gold.load_controller where schema_name = '{_schema_name}' and table_name = '{_table_name}' order by created_on desc").show()

+------+-----------+----------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
| layer|schema_name|table_name|load_type|write_mode|      load_timestamp|load_status|loaded_count|          created_on|          created_by|         modified_on|         modified_by|
+------+-----------+----------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
|silver|     silver|  customer|full-load| overwrite|2026-01-29 00:52:...|    success|           5|2026-01-29 00:52:...|silver_customer.i...|2026-01-29 00:52:...|silver_customer.i...|
+------+-----------+----------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+



In [18]:
from delta import DeltaTable

dt = DeltaTable.forName(spark,f"{_schema_name}.{_table_name}")##"bronze.dim_date_bronze")

dt.history().limit(1).select("version","operationMetrics.numOutputRows").show(truncate=False)

+-------+-------------+
|version|numOutputRows|
+-------+-------------+
|0      |5            |
+-------+-------------+



In [19]:
spark.stop()